In [1]:
# import packages
using BlockArrays
using LinearAlgebra
using UnPack
using LinearSolve 
using IncompleteLU
using SparseArrays
using Ferrite
using FerriteGmsh 
using OrdinaryDiffEq
using DifferentialEquations
using Plots 
using WriteVTK
using IterativeSolvers
using Interpolations
using LaTeXStrings
using ADTypes
using DiffEqBase
using SciMLBase
using Sundials

In [2]:
# declare variables
Ca = 59.187
Ea = 21179.6
eps = 0.5
rho_emp = 7164
rho_sat = 7259
R = 8.314
Aa = 10.7
Ba = 3704.6
Ta = 301.15
Pref = 10
Peqa = Pref * exp((Aa - Ba / Ta))
C = 1 / (R * Ta)
mdot = Ca * exp(-Ea / (R * Ta))
L = 0.2 #0.8
H = 0.04
viscosity = 8.9e-6
K = 1e-9
damping = viscosity / K
Ptot = 1.1 * (1 / C * (rho_sat - rho_emp));

In [3]:
# initialize grid
nels = (10, 10) 
left = Vec((0., 0.))  
right = Vec((L, H,))   
grid = generate_grid(Quadrilateral, nels, left, right);

In [4]:
# define structs
struct RHSparams{KT, MT, CH, DH, CVU, CVP, CVS, FVU, UT, JT}
    K::KT  
    M::MT  
    ch::CH 
    dh::DH 
    cvu::CVU 
    cvp::CVP 
    cvs::CVS 
    fvu::FVU
    u::UT  
    J_sparse::JT  
end

struct FreeDofErrorNorm
    ch::ConstraintHandler
end

In [5]:
# define assembly functions
function assemble_mass!(cellvalues_v::CellValues, cellvalues_p::CellValues, cellvalues_s::CellValues, M::SparseMatrixCSC, dh::DofHandler, u_old)
    # initialize
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs_s = getnbasefunctions(cellvalues_s)

    n_basefuncs = n_basefuncs_v + n_basefuncs_p + n_basefuncs_s
    
    v▄, p▄, s▄ = 1, 2, 3

    u_range = dof_range(dh, :u)
    p_range = dof_range(dh, :p)

    ndofs_u = length(u_range)
    ndofs_p = length(p_range)

    uₑ = zeros(ndofs_u)
    pₑ = zeros(ndofs_p)

    Mₑ = BlockArray(zeros(n_basefuncs, n_basefuncs), [n_basefuncs_v, n_basefuncs_p, n_basefuncs_s], [n_basefuncs_v, n_basefuncs_p, n_basefuncs_s])

    mass_assembler = start_assemble(M)

    # loop over cells
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)

        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)
        Ferrite.reinit!(cellvalues_s, cell)

        u_celldofs = @view celldofs(cell)[u_range]
        p_celldofs = @view celldofs(cell)[p_range]

        uₑ .= @views u_old[u_celldofs]
        pₑ .= @views u_old[p_celldofs]

        # loop over quadrature points
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)

            u = function_value(cellvalues_v, q_point, uₑ)
            p = function_value(cellvalues_p, q_point, pₑ)

            # u-u
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += C * p * φᵢ ⋅ φⱼ * dΩ
                end
            end
            # u-p
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_p
                    φⱼ = shape_value(cellvalues_p, q_point, j)
                    Mₑ[BlockIndex((v▄, p▄), (i, j))] += C * u ⋅ φᵢ * φⱼ * dΩ
                end
            end
            # p-p
            for i in 1:n_basefuncs_p
                φᵢ = shape_value(cellvalues_p, q_point, i)
                for j in 1:n_basefuncs_p
                    φⱼ = shape_value(cellvalues_p, q_point, j)
                    Mₑ[BlockIndex((p▄, p▄), (i, j))] += eps * C * φᵢ * φⱼ * dΩ
                end
            end
            # s-s
            for i in 1:n_basefuncs_s
                φᵢ = shape_value(cellvalues_s, q_point, i)
                for j in 1:n_basefuncs_s
                    φⱼ = shape_value(cellvalues_s, q_point, j)
                    Mₑ[BlockIndex((s▄, s▄), (i, j))] += (1 - eps) * φᵢ * φⱼ * dΩ
                end
            end
        end
        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    return M
end

function assemble_linear!(K, dh, cvu, cvp, cvs, viscosity, damping)
    # initialize
    range_u = dof_range(dh, :u)
    range_p = dof_range(dh, :p)
    range_s = dof_range(dh, :s)
    
    ndofs_u = length(range_u)
    ndofs_p = length(range_p)
    ndofs_s = length(range_s)

    ϕᵤ = Vector{Vec{2,Float64}}(undef, ndofs_u)
    ∇ϕᵤ = Vector{Tensor{2,2,Float64,4}}(undef, ndofs_u) 
    divϕᵤ = Vector{Float64}(undef, ndofs_u)
    ϕₚ = Vector{Float64}(undef, ndofs_p)
    ϕₛ = Vector{Float64}(undef, ndofs_s)

    ke = zeros(ndofs_per_cell(dh), ndofs_per_cell(dh))

    assembler = start_assemble(K)

    # loop over cells
    for cell in CellIterator(dh)
        ke .= 0

        Ferrite.reinit!(cvu, cell)
        Ferrite.reinit!(cvp, cell)
        Ferrite.reinit!(cvs, cell)

        coords = getcoordinates(cell)

        # loop over quadrature points
        for qp in 1:getnquadpoints(cvu)
            dΩ = getdetJdV(cvu, qp)

            for i in 1:ndofs_u
                ϕᵤ[i] = shape_value(cvu, qp, i)
                ∇ϕᵤ[i] = shape_gradient(cvu, qp, i)
                divϕᵤ[i] = shape_divergence(cvu, qp, i)
            end

            for i in 1:ndofs_p
                ϕₚ[i] = shape_value(cvp, qp, i)
            end

            for i in 1:ndofs_s
                ϕₛ[i] = shape_value(cvs, qp, i)
            end
            
            # u-u
            for (i, I) in pairs(range_u), (j, J) in pairs(range_u)
                ke[I, J] += viscosity*( ∇ϕᵤ[i] ⊡ ∇ϕᵤ[j] ) * dΩ + damping*( ϕᵤ[i]⋅ϕᵤ[j] ) * dΩ 
            end
            # u-p
            for (i, I) in pairs(range_u), (j, J) in pairs(range_p)
                ke[I, J] += ( -divϕᵤ[i] * ϕₚ[j] ) * dΩ
            end
        end
        assemble!(assembler, celldofs(cell), ke)
    end
    return K 
end

function assemble_nonlinear!(dpₑ, pₑ, dsₑ, sₑ, uₑ, cvp, cvu, cvs)
    #initialize
    n_basefuncs_p = getnbasefunctions(cvp)
    n_basefuncs_s = getnbasefunctions(cvs)

    # loop over quadrature points
    for q_point in 1:getnquadpoints(cvp)
        dΩ = getdetJdV(cvp, q_point)

        u = function_value(cvu, q_point, uₑ)
        div_u = function_divergence(cvu, q_point, uₑ)
        p = function_value(cvp, q_point, pₑ)
        p_safe = max(p, 1e-8)
        ∇p = function_gradient(cvp, q_point, pₑ)
        s = function_value(cvs, q_point, sₑ)

        # p
        for j in 1:n_basefuncs_p
            φⱼ = shape_value(cvp, q_point, j)
            dpₑ[j] -= - C * φⱼ * (dot(u, ∇p) + p * div_u) * dΩ + mdot * φⱼ * log(p_safe / Peqa) * (rho_sat - s) * dΩ
        end
        # s
        for j in 1:n_basefuncs_s
            φⱼ = shape_value(cvs, q_point, j)
            dsₑ[j] -= mdot * φⱼ * log(p_safe / Peqa) * (rho_sat - s) * dΩ
        end
    end
    return
end

function assemble_inlet!(f, dh, fvu, inlet, u_old, Ptot)
    # initialize
    u_range = dof_range(dh, :u)

    # loop over cells corresponding to inlet
    for facet in FacetIterator(dh, inlet)
        Ferrite.reinit!(fvu, facet)

        cell_dofs = celldofs(facet)
        cell_u_dofs = cell_dofs[u_range]

        ue = u_old[cell_u_dofs]

        # loop over quadrature points
        for q in 1:Ferrite.getnquadpoints(fvu)
            dΓ = Ferrite.getdetJdV(fvu, q)
            n = Ferrite.getnormal(fvu, q)
            uh = Ferrite.function_value(fvu, q, ue)

            p_bc = max(Ptot - 0.5 * dot(uh, uh), 0.0)
            traction = - p_bc * n

            for i in 1:Ferrite.getnbasefunctions(fvu)
                δu = Ferrite.shape_value(fvu, q, i)
                f[cell_u_dofs[i]] += dot(δu, traction) * dΓ
            end
        end
    end
    return f
end;

In [6]:
# define jacobian functions
function assemble_element_jacobian!(Jppₑ, Jpuₑ, Jpsₑ, Jspₑ, Jssₑ, pₑ, uₑ, sₑ, cvp, cvu, cvs)
    # initialize
    n_basefuncs_p = getnbasefunctions(cvp)
    n_basefuncs_u = getnbasefunctions(cvu)
    n_basefuncs_s = getnbasefunctions(cvs)

    # loop over quadrature points
    for q_point in 1:getnquadpoints(cvp)
        dΩ = getdetJdV(cvp, q_point)

        u = function_value(cvu, q_point, uₑ)
        div_u = function_divergence(cvu, q_point, uₑ)
        p = function_value(cvp, q_point, pₑ)
        p_safe = max(p, 1e-8)
        ∇p = function_gradient(cvp, q_point, pₑ)
        s = function_value(cvs, q_point, sₑ)

        # p
        for j in 1:n_basefuncs_p
            φⱼ = shape_value(cvp, q_point, j)
            # p-p
            for i in 1:n_basefuncs_p
                φᵢ = shape_value(cvp, q_point, i)
                ∇φᵢ = shape_gradient(cvp, q_point, i)
                Jppₑ[j, i] -= - C * φⱼ * (dot(u, ∇φᵢ) + φᵢ * div_u) * dΩ + mdot * φⱼ * φᵢ * (rho_sat - s) / p_safe * dΩ
            end
            # p-u
            for i in 1:n_basefuncs_u
                φᵢ = shape_value(cvu, q_point, i)
                div_φᵢ = shape_divergence(cvu, q_point, i)
                Jpuₑ[j, i] += C * φⱼ * (dot(φᵢ, ∇p) + p * div_φᵢ) * dΩ
            end
            # p-s
            for i in 1:n_basefuncs_s
                φᵢ = shape_value(cvs, q_point, i)
                Jpsₑ[j, i] += mdot * φⱼ * log(p_safe / Peqa) * φᵢ * dΩ
            end
        end
        # s
        for j in 1:n_basefuncs_s
            φⱼ = shape_value(cvs, q_point, j)
            # s-p
            for i in 1:n_basefuncs_p
                φᵢ = shape_value(cvp, q_point, i)
                Jspₑ[j, i] -= mdot * φⱼ * φᵢ * (rho_sat - s) / p_safe * dΩ
            end
            # s-s
            for i in 1:n_basefuncs_s
                φᵢ = shape_value(cvs, q_point, i)
                Jssₑ[j, i] += mdot * φⱼ * log(p_safe / Peqa) * φᵢ * dΩ
            end
        end
    end
    return
end

function assemble_inlet_jacobian!(J, dh, fvu, inlet, u)
    # initialize
    u_range = dof_range(dh, :u)

    ndofs_u = length(u_range)

    Ke = zeros(ndofs_u, ndofs_u)

    assembler = start_assemble(J; fillzero=false)

    # loop over cells corresponding to inlet
    for facet in FacetIterator(dh, inlet)
        fill!(Ke, 0.0)

        Ferrite.reinit!(fvu, facet)

        cell_dofs = celldofs(facet)
        cell_u_dofs = cell_dofs[u_range]

        ue = u[cell_u_dofs]

        # loop over quadrature points
        for q in 1:getnquadpoints(fvu)
            dΓ = getdetJdV(fvu, q)
            n = getnormal(fvu, q)
            uh = function_value(fvu, q, ue)

            for i in 1:getnbasefunctions(fvu)
                ϕi = shape_value(fvu, q, i)
                for j in 1:getnbasefunctions(fvu)
                    ϕj = shape_value(fvu, q, j)
                    Ke[i,j] += dot(uh, ϕj) * dot(ϕi, n) * dΓ
                end
            end
        end
        assemble!(assembler, cell_u_dofs, cell_u_dofs, Ke)
    end
    return
end

function assemble_jacobian!(J, u_uc, params, t)
    # unpack parameters
    (; K, ch, dh, cvu, cvp, cvs, fvu, u, J_sparse) = params

    # update constraint handler
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)

    # account for linear contribution 
    fill!(J_sparse, 0.0)
    copyto!(J_sparse, K)

    # account for nonlinear contribution 
    assembler = start_assemble(J_sparse; fillzero = false)

    u_range = dof_range(dh, :u)
    p_range = dof_range(dh, :p)
    s_range = dof_range(dh, :s)

    ndofs_u = length(u_range)
    ndofs_p = length(p_range)
    ndofs_s = length(s_range)

    uₑ = zeros(ndofs_u)
    pₑ = zeros(ndofs_p)
    sₑ = zeros(ndofs_s)

    Jppₑ = zeros(ndofs_p, ndofs_p)
    Jpuₑ = zeros(ndofs_p, ndofs_u)
    Jpsₑ = zeros(ndofs_p, ndofs_s)
    Jspₑ = zeros(ndofs_s, ndofs_p)
    Jssₑ = zeros(ndofs_s, ndofs_s)

    for cell in CellIterator(dh)
        Ferrite.reinit!(cvu, cell)
        Ferrite.reinit!(cvp, cell)
        Ferrite.reinit!(cvs, cell)

        u_celldofs = @view celldofs(cell)[u_range]
        p_celldofs = @view celldofs(cell)[p_range]
        s_celldofs = @view celldofs(cell)[s_range]
       
        uₑ .= @views u[u_celldofs]
        pₑ .= @views u[p_celldofs]
        sₑ .= @views u[s_celldofs]
       
        fill!(Jppₑ, 0.0)
        fill!(Jpuₑ, 0.0)
        fill!(Jpsₑ, 0.0)
        fill!(Jspₑ, 0.0)
        fill!(Jssₑ, 0.0)

        assemble_element_jacobian!(Jppₑ, Jpuₑ, Jpsₑ, Jspₑ, Jssₑ, pₑ, uₑ, sₑ, cvp, cvu, cvs)

        assemble!(assembler, p_celldofs, p_celldofs, Jppₑ)
        assemble!(assembler, p_celldofs, u_celldofs, Jpuₑ)
        assemble!(assembler, p_celldofs, s_celldofs, Jpsₑ)
        assemble!(assembler, s_celldofs, p_celldofs, Jspₑ)
        assemble!(assembler, s_celldofs, s_celldofs, Jssₑ)
    end
    
    # account for inlet boundary condition
    assemble_inlet_jacobian!(J_sparse, dh, fvu, inlet, u)

    apply!(J_sparse, ch)
    copyto!(J, J_sparse)
    return 
end;

In [7]:
# define update function
function assemble_update!(f, u_uc, params, t)
    # unpack parameters
    (; K, M, ch, dh, cvu, cvp, cvs, fvu, u) = params

    # update constraint handler
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)

    # update mass matrix
    fill!(M, 0.0)
    assemble_mass!(cvu, cvp, cvs, M, dh, u)
    apply!(M, ch)

    # account for linear contribution
    fill!(f, 0.0)
    mul!(f, K, u)   

    # account for nonlinear contribution
    u_range = dof_range(dh, :u)
    p_range = dof_range(dh, :p)
    s_range = dof_range(dh, :s)

    ndofs_u = length(u_range)
    ndofs_p = length(p_range)
    ndofs_s = length(s_range)

    uₑ = zeros(ndofs_u)
    pₑ = zeros(ndofs_p)
    sₑ = zeros(ndofs_s)
    dpₑ = zeros(ndofs_p)
    dsₑ = zeros(ndofs_s)

    for cell in CellIterator(dh)
        Ferrite.reinit!(cvu, cell)
        Ferrite.reinit!(cvp, cell)
        Ferrite.reinit!(cvs, cell)

        u_celldofs = @view celldofs(cell)[u_range]
        p_celldofs = @view celldofs(cell)[p_range]
        s_celldofs = @view celldofs(cell)[s_range]

        uₑ .= @views u[u_celldofs]
        pₑ .= @views u[p_celldofs]
        sₑ .= @views u[s_celldofs]

        fill!(dpₑ, 0.0)
        fill!(dsₑ, 0.0)

        assemble_nonlinear!(dpₑ, pₑ, dsₑ, sₑ, uₑ, cvp, cvu, cvs)

        assemble!(f, p_celldofs, dpₑ)
        assemble!(f, s_celldofs, dsₑ)
    end
    # account for inlet boundary condition
    assemble_inlet!(f, dh, fvu, inlet, u, Ptot)
    apply_zero!(f, ch)
    return 
end;

In [8]:
# initialize FEM values
dim = 2
degree = 2

ipu = Lagrange{RefQuadrilateral,degree+1}() ^ dim
ipp = Lagrange{RefQuadrilateral,degree}()
ips = Lagrange{RefQuadrilateral,degree}()

dh = DofHandler(grid)
add!(dh, :u, ipu)
add!(dh, :p, ipp)
add!(dh, :s, ips)
close!(dh)

qr = QuadratureRule{RefQuadrilateral}(2*degree+1)
ipg = Lagrange{RefQuadrilateral,1}()

cvu = CellValues(qr, ipu, ipg)
cvp = CellValues(qr, ipp, ipg)
cvs = CellValues(qr, ips, ipg)

u_range = dof_range(dh, :u)
p_range = dof_range(dh, :p)
s_range = dof_range(dh, :s)

ndofs_u = length(u_range)
ndofs_p = length(p_range)
ndofs_s = length(s_range);

In [9]:
# set boundary conditions
ch = ConstraintHandler(dh)

inlet = getfacetset(grid, "left")
zero_u_boundary = union(getfacetset(grid, "right"), getfacetset(grid, "top"), getfacetset(grid, "bottom"))
dbc_u0 = Dirichlet(:u, zero_u_boundary, (x, t) -> [0.0, 0.0])
add!(ch, dbc_u0)

close!(ch)
update!(ch, 0.0)

facet_qr = FacetQuadratureRule{RefQuadrilateral}(2 * degree + 1)
fvu = FacetValues(facet_qr, ipu, ipg);

In [10]:
# set initial conditions
un = zeros(ndofs(dh))
for facet in FacetIterator(dh, inlet)
        Ferrite.reinit!(fvu, facet)

        cell_dofs = celldofs(facet)
        u_dofs = cell_dofs[u_range]

        for i in 1:2:length(u_dofs)
            un[u_dofs[i]] = sqrt(2 * Ptot)
            un[u_dofs[i+1]] = 0.0   
        end
end
for cell in CellIterator(dh)
    s_dofs = celldofs(cell)[s_range]
    un[s_dofs] .= rho_emp
end
dun = zeros(ndofs(dh))

apply!(un, ch)

rm.(filter(f -> startswith(f, "stokes_and_densities_absorption"), readdir()))
pvd = paraview_collection("stokes_and_densities_absorption")
VTKGridFile("stokes_and_densities_absorption-0", dh) do vtk
    write_solution(vtk, dh, round.(un; digits=6))
    pvd[0.0] = vtk
end;

In [11]:
# initialize matrices and parameters
M = allocate_matrix(dh, ch)
M = assemble_mass!(cvu, cvp, cvs, M, dh, un)
apply!(M, ch)

K = allocate_matrix(dh, ch)
K = assemble_linear!(K, dh, cvu, cvp, cvs, viscosity, damping)

jac_sparsity = sparse(K)
params = RHSparams(K, M, ch, dh, cvu, cvp, cvs, fvu, copy(un), jac_sparsity);

In [ ]:
# solve problem
T  = 100.
tspan = (0.0, T)

func = ODEFunction(assemble_update!; jac = assemble_jacobian!, jac_prototype = jac_sparsity, mass_matrix = M)
prob = ODEProblem(func, un, tspan, params)

(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

integrator = init(prob, Rodas5P(autodiff = false), abstol = 1e-5, reltol = 1e-5, adaptive = true)

i = 0
while integrator.t < T
    step!(integrator)

    i += 1

    uh = integrator.u
    t  = integrator.t

    VTKGridFile("stokes_and_densities_absorption-$i", dh) do vtk
        write_solution(vtk, dh, uh)
        pvd[t] = vtk
    end

    println("step = $i, t = $t, dt = $(integrator.dt)")
end

vtk_save(pvd);